# 01 — Build Corpus and FAISS Index

This notebook builds and evaluates the retrieval infrastructure for the **Multi-hop RAG** project using HotpotQA.

## Pipeline

HotpotQA  
→ Sample Selection  
→ Corpus Construction  
→ Text Embeddings  
→ FAISS Index  
→ Dense Retrieval  
→ Retrieval Evaluation

## Dataset

HotpotQA is loaded directly from Hugging Face.

For this pipeline setup:
- All validation split samples are selected (7,405 samples total).
- 30 samples are held out for evaluating retrieval performance.
- Each HotpotQA context paragraph is treated as one retrieval chunk, resulting in a 73,700-passage corpus.
- Supporting facts are preserved in the dataset for evaluation only and are not provided to the retriever.

## Embedding Model

`sentence-transformers/all-MiniLM-L6-v2`

## Vector Index

FAISS `IndexFlatIP` is used with normalized embeddings. Since the embeddings are normalized, inner product searches correspond to cosine similarity.

## Evaluation

Our retrieval evaluation uses:
- Recall@5
- Retrieval Success Rate

## Project Configuration

Project artifacts are stored under `Multi_hop_RAG` in Google Drive.

The Google Drive path is used as the runtime storage location in Google Colab.
Generated datasets, corpus files, embeddings, FAISS indexes, and evaluation
results are not intended to be committed to GitHub.

In [17]:
# ============================================================
# Cell 1 — Project Configuration
# ============================================================

# Google Drive project root
PROJECT_ROOT = "/content/drive/MyDrive/Multi_hop_RAG"

# Dataset
DATASET_NAME = "hotpotqa"
DATASET_DIR = f"{PROJECT_ROOT}/data/{DATASET_NAME}"

# Corpus and index
CORPUS_DIR = f"{PROJECT_ROOT}/corpus/{DATASET_NAME}"
INDEX_DIR = f"{PROJECT_ROOT}/indexes/{DATASET_NAME}"

# Evaluation results
RESULTS_DIR = f"{PROJECT_ROOT}/results/{DATASET_NAME}"

# Notebooks
NOTEBOOK_DIR = f"{PROJECT_ROOT}/notebooks"

# Reproducibility
RANDOM_SEED = 42

# Dataset size for this first experiment
N_SAMPLES = 7405
N_EVAL = 30

# Retrieval setting used later in Simple RAG
DEFAULT_TOP_K = 5

print("✓ Configuration loaded.")
print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset      : {DATASET_NAME}")
print(f"Samples      : {N_SAMPLES}")
print(f"Evaluation   : {N_EVAL}")
print(f"Top-k        : {DEFAULT_TOP_K}")

✓ Configuration loaded.
Project root : /content/drive/MyDrive/Multi_hop_RAG
Dataset      : hotpotqa
Samples      : 7405
Evaluation   : 30
Top-k        : 5


## Dependencies

Install the Python packages required for dataset loading, text embedding,
and FAISS-based vector retrieval.

In [18]:
# ============================================================
# Cell 2 — Install Dependencies
# ============================================================

!pip install -q -U datasets sentence-transformers faiss-cpu

print("✓ Dependencies installed.")


✓ Dependencies installed.


## Google Drive and project directories

mount Drive and create the project directories.

In [19]:
# ============================================================
# Cell 3 — Mount Google Drive & Create Project Directories
# ============================================================

from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive")

# Create project directories
directories = [
    PROJECT_ROOT,
    DATASET_DIR,
    CORPUS_DIR,
    INDEX_DIR,
    RESULTS_DIR,
    NOTEBOOK_DIR,
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)

print("✓ Google Drive mounted.")
print("✓ Project directories are ready.\n")

print("Project structure:")
print(f"  {PROJECT_ROOT}/")
print(f"  ├── data/{DATASET_NAME}/")
print(f"  ├── corpus/{DATASET_NAME}/")
print(f"  ├── indexes/{DATASET_NAME}/")
print(f"  ├── results/{DATASET_NAME}/")
print(f"  └── notebooks/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted.
✓ Project directories are ready.

Project structure:
  /content/drive/MyDrive/Multi_hop_RAG/
  ├── data/hotpotqa/
  ├── corpus/hotpotqa/
  ├── indexes/hotpotqa/
  ├── results/hotpotqa/
  └── notebooks/


## Load HotpotQA

Load the **distractor** version of HotpotQA directly from Hugging Face.

No local copy of the dataset is required.

In [20]:
# ============================================================
# Cell 4 — Load HotpotQA
# ============================================================

from datasets import load_dataset

# Load the distractor version of HotpotQA
hotpotqa = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor"
)

print("✓ HotpotQA loaded successfully.\n")

print("Dataset structure:")
print(hotpotqa)

print("\nAvailable splits:")
print(hotpotqa.keys())

✓ HotpotQA loaded successfully.

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 90447
    })
    validation: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 7405
    })
})

Available splits:
dict_keys(['train', 'validation'])


## Select Samples

Select all 7,405 validation samples from HotpotQA to build our corpus. Then, randomly select 30 dedicated samples to evaluate the FAISS retrieval performance, using a fixed random seed for reproducibility.

In [21]:
# ============================================================
# Cell 5 — Select Samples
# ============================================================

import random

# Use the validation split for our development dataset
validation_data = hotpotqa["validation"]

# Create a reproducible list of indices
indices = list(range(len(validation_data)))

random.seed(RANDOM_SEED)
random.shuffle(indices)

# Select All samples
selected_indices = indices[:]

# Select 30 evaluation samples from the samples
eval_indices = selected_indices[:N_EVAL]

print(f"Total validation samples : {len(validation_data)}")
print(f"Selected samples         : {len(selected_indices)}")
print(f"Evaluation samples       : {len(eval_indices)}")


Total validation samples : 7405
Selected samples         : 7405
Evaluation samples       : 30


## Save Selected and Evaluation Samples


In [22]:
# ============================================================
# Cell 6 — Save/Load Selected Samples
# ============================================================

import json
import os

selected_path = f"{DATASET_DIR}/all_evaluation_samples.json"
evaluation_path = f"{DATASET_DIR}/evaluation_30.json"

if os.path.exists(selected_path) and os.path.exists(evaluation_path):
    print("✓ Selected samples and evaluation samples found on Drive. Loading files...")
    with open(selected_path, "r", encoding="utf-8") as f:
        selected_samples = json.load(f)
    with open(evaluation_path, "r", encoding="utf-8") as f:
        evaluation_samples = json.load(f)
else:
    print("Creating new selected samples...")
    # Convert selected indices to actual dataset records
    selected_samples = [
        validation_data[i]
        for i in selected_indices
    ]

    evaluation_samples = [
        validation_data[i]
        for i in eval_indices
    ]

    # Save samples
    with open(selected_path, "w", encoding="utf-8") as f:
        json.dump(
            selected_samples,
            f,
            ensure_ascii=False,
            indent=2
        )

    # Save evaluation 30 samples
    with open(evaluation_path, "w", encoding="utf-8") as f:
        json.dump(
            evaluation_samples,
            f,
            ensure_ascii=False,
            indent=2
        )
    print("✓ All evaluation samples saved to Drive.")

print(f"Loaded counts:")
print(f"  Samples    : {len(selected_samples)}")
print(f"  Evaluation : {len(evaluation_samples)}")

✓ Selected samples and evaluation samples found on Drive. Loading files...
Loaded counts:
  Samples    : 7405
  Evaluation : 30


## Build Full Corpus

Extracting context paragraphs from the selected validation samples. Each paragraph is formatted as an independent passage chunk with unique metadata, preserving the ground-truth titles for evaluations.

In [23]:
# ============================================================
# Cell 7 — Build/Load Full Corpus
# ============================================================

import json
import os

corpus_path = f"{CORPUS_DIR}/corpus.json"

if os.path.exists(corpus_path):
    print("✓ Existing corpus file found on Drive. Loading...")
    with open(corpus_path, "r", encoding="utf-8") as f:
        corpus = json.load(f)
else:
    print("Building corpus...")
    corpus = []
    for sample in selected_samples:
        sample_id = sample["id"]

        titles = sample["context"]["title"]
        sentence_groups = sample["context"]["sentences"]

        for idx, (title, sentences) in enumerate(
            zip(titles, sentence_groups)
        ):
            # Each paragraph is one chunk
            text = " ".join(sentences).strip()

            chunk = {
                "chunk_id": f"{sample_id}_chunk_{idx}",
                "sample_id": sample_id,
                "title": title,
                "text": text,
                "dataset": DATASET_NAME,
                "source_type": "context"
            }
            corpus.append(chunk)

    # Save corpus
    with open(corpus_path, "w", encoding="utf-8") as f:
        json.dump(
            corpus,
            f,
            ensure_ascii=False,
            indent=2
        )
    print("✓ Corpus built and saved successfully.")

print(f"Total chunks: {len(corpus)}")

✓ Existing corpus file found on Drive. Loading...
Total chunks: 73700


## Corpus Sanity Check

In [24]:
# ============================================================
# Cell 8 — Corpus Sanity Check
# ============================================================

import statistics

# Check empty fields
empty_chunk_ids = [
    c["chunk_id"]
    for c in corpus
    if not c["chunk_id"]
]

empty_titles = [
    c["chunk_id"]
    for c in corpus
    if not c["title"]
]

empty_texts = [
    c["chunk_id"]
    for c in corpus
    if not c["text"].strip()
]

# Calculate text lengths
text_lengths = [
    len(c["text"])
    for c in corpus
]

print("Corpus sanity check")
print("=" * 60)

print(f"Total chunks       : {len(corpus)}")
print(f"Unique chunk IDs   : {len(set(c['chunk_id'] for c in corpus))}")

print(f"\nEmpty chunk IDs    : {len(empty_chunk_ids)}")
print(f"Empty titles       : {len(empty_titles)}")
print(f"Empty texts        : {len(empty_texts)}")

print("\nText length statistics:")
print(f"  Minimum          : {min(text_lengths)} characters")
print(f"  Maximum          : {max(text_lengths)} characters")
print(f"  Mean             : {statistics.mean(text_lengths):.1f} characters")
print(f"  Median           : {statistics.median(text_lengths):.1f} characters")

print("\nFirst 3 chunk titles:")
for chunk in corpus[:3]:
    print(f"  - {chunk['title']}")

Corpus sanity check
Total chunks       : 73700
Unique chunk IDs   : 73700

Empty chunk IDs    : 0
Empty titles       : 0
Empty texts        : 0

Text length statistics:
  Minimum          : 29 characters
  Maximum          : 8268 characters
  Mean             : 552.7 characters
  Median           : 489.0 characters

First 3 chunk titles:
  - Radar Station (film)
  - RAF Trimingham
  - NATO Ouvrage &quot;G&quot;


## Load embedding model

Model: `sentence-transformers/all-MiniLM-L6-v2`

In [25]:
# ============================================================
# Cell 9 — Load Embedding Model
# ============================================================

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("✓ Embedding model loaded.")
print(f"Model: {EMBEDDING_MODEL_NAME}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Embedding model loaded.
Model: sentence-transformers/all-MiniLM-L6-v2


## Create normalized embeddings

encode corpus texts and normalize embeddings.

In [26]:
# ============================================================
# Cell 10 — Generate/Load Normalized Embeddings
# ============================================================

import numpy as np
import os

drive_embeddings_path = f"{INDEX_DIR}/embeddings.npy"

# Extract chunk texts
corpus_texts = [
    chunk["text"]
    for chunk in corpus
]

if os.path.exists(drive_embeddings_path):
    print("✓ Existing embeddings found on Drive! Loading embeddings...")
    embeddings = np.load(drive_embeddings_path)
else:
    print("Embeddings not found on Drive. Generating new normalized embeddings (this may take a few minutes)...")
    # Generate normalized embeddings
    embeddings = embedding_model.encode(
        corpus_texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    # Convert to NumPy array
    embeddings = np.asarray(embeddings, dtype="float32")
    print("✓ Embeddings generated.")

print(f"Number of embeddings : {embeddings.shape[0]}")
print(f"Embedding dimension  : {embeddings.shape[1]}")
print(f"Embedding shape      : {embeddings.shape}")

✓ Existing embeddings found on Drive! Loading embeddings...
Number of embeddings : 73700
Embedding dimension  : 384
Embedding shape      : (73700, 384)


## Save Generated Embeddings

Save the newly generated corpus embeddings to Google Drive. To bypass slow write speeds on mounted Google Drive runtimes, we first write the array to a local temporary file and then copy it over.

In [27]:
# ============================================================
# Cell 11 — Save Embeddings to Google Drive
# ============================================================

import numpy as np
import os
import shutil

drive_embeddings_path = f"{INDEX_DIR}/embeddings.npy"
local_embeddings_path = "/content/embeddings_temp.npy"

if os.path.exists(drive_embeddings_path):
    print("✓ Embeddings already exist on Google Drive. Skipping save step.")
else:
    print("Saving embeddings locally...")
    np.save(local_embeddings_path, embeddings)
    print(f"✓ Saved locally to {local_embeddings_path}")

    print(f"Copying local file to Google Drive: {drive_embeddings_path}...")
    shutil.copy2(local_embeddings_path, drive_embeddings_path)
    print("✓ Copy to Google Drive complete!")

# Verify the file on Google Drive
if os.path.exists(drive_embeddings_path):
    verified_embeddings = np.load(drive_embeddings_path)
    print("\nVerification:")
    print(f"  Path  : {drive_embeddings_path}")
    print(f"  Shape : {verified_embeddings.shape}")
    print(f"  Dtype : {verified_embeddings.dtype}")
else:
    print("⚠ Error: Embeddings file not found on Google Drive!")

✓ Embeddings already exist on Google Drive. Skipping save step.

Verification:
  Path  : /content/drive/MyDrive/Multi_hop_RAG/indexes/hotpotqa/embeddings.npy
  Shape : (73700, 384)
  Dtype : float32


## Build FAISS Index

use `faiss.IndexFlatIP` over normalized embeddings.

In [28]:
# ============================================================
# Cell 12 — Build FAISS Index
# ============================================================

import faiss
import os

index_path = f"{INDEX_DIR}/faiss.index"

if os.path.exists(index_path):
    print("✓ FAISS index already exists on Google Drive. Skipping build step.")
    print("Loading existing FAISS index to verify...")
    index = faiss.read_index(index_path)
else:
    print("FAISS index not found on Google Drive. Building new index...")
    # Embedding dimension
    embedding_dim = embeddings.shape[1]

    # Create Inner Product index
    index = faiss.IndexFlatIP(embedding_dim)

    # Add normalized embeddings to the index
    index.add(embeddings)
    print("✓ FAISS index built.")

print(f"\nIndex type       : {type(index).__name__}")
print(f"Embedding dim    : {index.d}")
print(f"Number of vectors: {index.ntotal}")

✓ FAISS index already exists on Google Drive. Skipping build step.
Loading existing FAISS index to verify...

Index type       : IndexFlatIP
Embedding dim    : 384
Number of vectors: 73700


## Save FAISS Index


In [29]:
# ============================================================
# Cell 13 — Save FAISS Index
# ============================================================

import faiss
import os

index_path = f"{INDEX_DIR}/faiss.index"

if os.path.exists(index_path):
    print("✓ FAISS index already exists on Google Drive. Skipping save step.")
else:
    print(f"Saving FAISS index to Google Drive: {index_path}...")
    faiss.write_index(index, index_path)
    print("✓ FAISS index saved.")

# Verify the saved index
if os.path.exists(index_path):
    loaded_index = faiss.read_index(index_path)
    print("\nVerification:")
    print(f"  Index type        : {type(loaded_index).__name__}")
    print(f"  Number of vectors : {loaded_index.ntotal}")
    print(f"  Embedding dim     : {loaded_index.d}")
else:
    print("⚠ Error: FAISS index file not found on Google Drive!")

✓ FAISS index already exists on Google Drive. Skipping save step.

Verification:
  Index type        : IndexFlatIP
  Number of vectors : 73700
  Embedding dim     : 384


## Retrieval smoke test

A small retrieval test to verify that the saved index can be queried.

In [30]:
# ============================================================
# Cell 14 — Retrieval Smoke Test
# ============================================================

# Select one evaluation question
test_sample = evaluation_samples[10]

query = test_sample["question"]

print("Query:")
print(query)

print("\n" + "=" * 80)

# Encode the query
query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

query_embedding = np.asarray(
    query_embedding,
    dtype="float32"
)

# Retrieve top-k chunks
scores, indices = index.search(
    query_embedding,
    DEFAULT_TOP_K
)

print(f"Top-{DEFAULT_TOP_K} retrieved chunks:\n")

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):
    chunk = corpus[idx]

    print(f"Rank {rank}")
    print(f"Score    : {score:.4f}")
    print(f"Chunk ID : {chunk['chunk_id']}")
    print(f"Title    : {chunk['title']}")
    print(f"Text     : {chunk['text'][:300]}...")
    print("-" * 80)


Query:
Which Captain launched the attack which led to more casualties than any other incident in the war fought between the settlers of the nascent colony of New Netherland and the native Lenape population?

Top-5 retrieved chunks:

Rank 1
Score    : 0.6446
Chunk ID : 5ae4c01e55429913cc2044f3_chunk_2
Title    : Kieft's War
Text     : Kieft's War, also known as the Wappinger War, was a conflict (1643–1645) between settlers of the nascent colony of New Netherland and the native Lenape population in what would later become the New York metropolitan area of the United States.  It is named for Director-General of New Netherland Wille...
--------------------------------------------------------------------------------
Rank 2
Score    : 0.5066
Chunk ID : 5a75e67d5542992db9473709_chunk_3
Title    : Battle of Mamaroneck
Text     : The Battle of Mamaroneck was a skirmish in the New York and New Jersey campaign of the American Revolutionary War fought on October 22, 1776, at Mamaroneck, Westcheste

## Check Retrieved Chunks Against Supporting Facts

Compare the retrieved chunks with the gold supporting facts for one evaluation
sample.

Supporting facts are used here **only for evaluation** and are not provided
to the retriever.

In [31]:
# ============================================================
# Cell 15 — Check Retrieved Chunks Against Supporting Facts
# ============================================================

# Gold supporting facts
supporting_facts = test_sample["supporting_facts"]

gold_titles = set(
    supporting_facts["title"]
)

print("Gold supporting fact titles:")
for title in gold_titles:
    print(f"  - {title}")

print("\n" + "=" * 80)

# Retrieved chunks
retrieved_chunks = [
    corpus[idx]
    for idx in indices[0]
]

print("Retrieved titles:")
for rank, chunk in enumerate(retrieved_chunks, start=1):
    is_gold = chunk["title"] in gold_titles

    print(
        f"Rank {rank}: "
        f"{chunk['title']} "
        f"--> {'✓ GOLD EVIDENCE' if is_gold else 'not gold evidence'}"
    )

Gold supporting fact titles:
  - Pound Ridge massacre
  - Kieft's War

Retrieved titles:
Rank 1: Kieft's War --> ✓ GOLD EVIDENCE
Rank 2: Battle of Mamaroneck --> not gold evidence
Rank 3: Pound Ridge massacre --> ✓ GOLD EVIDENCE
Rank 4: Second Battle of Fort Wagner --> not gold evidence
Rank 5: East Jeddore, Nova Scotia --> not gold evidence


## Reusable FAISS Retriever


In [32]:
# ============================================================
# Cell 16 — Reusable FAISS Retriever
# ============================================================

class FAISSRetriever:
    """
    Simple reusable retriever based on:
    SentenceTransformer + FAISS IndexFlatIP
    """

    def __init__(self, embedding_model, index, corpus):
        self.embedding_model = embedding_model
        self.index = index
        self.corpus = corpus

    def retrieve(self, query, top_k=5):
        """
        Retrieve the top-k most similar chunks for a query.
        """

        # Encode query
        query_embedding = self.embedding_model.encode(
            [query],
            normalize_embeddings=True
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype="float32"
        )

        # Search FAISS
        scores, indices = self.index.search(
            query_embedding,
            top_k
        )

        # Build readable results
        results = []

        for rank, (score, idx) in enumerate(
            zip(scores[0], indices[0]),
            start=1
        ):
            chunk = self.corpus[idx]

            result = {
                "rank": rank,
                "score": float(score),
                "chunk_id": chunk["chunk_id"],
                "sample_id": chunk["sample_id"],
                "title": chunk["title"],
                "text": chunk["text"],
                "dataset": chunk["dataset"],
                "source_type": chunk["source_type"]
            }

            results.append(result)

        return results


# Create retriever
retriever = FAISSRetriever(
    embedding_model=embedding_model,
    index=index,
    corpus=corpus
)

print("✓ Retriever created successfully.")


✓ Retriever created successfully.


## Test Reusable Retriever

In [33]:
# ============================================================
# Cell 17 — Test Reusable Retriever
# ============================================================

test_sample = evaluation_samples[10]

query = test_sample["question"]

results = retriever.retrieve(
    query,
    top_k=DEFAULT_TOP_K
)

print("Query:")
print(query)

print("\n" + "=" * 80)

print(f"Top-{DEFAULT_TOP_K} Retrieved Chunks:\n")

for result in results:
    print(f"Rank {result['rank']}")
    print(f"Score    : {result['score']:.4f}")
    print(f"Chunk ID : {result['chunk_id']}")
    print(f"Sample ID: {result['sample_id']}")
    print(f"Title    : {result['title']}")
    print(f"Text     : {result['text'][:300]}...")
    print("-" * 80)

Query:
Which Captain launched the attack which led to more casualties than any other incident in the war fought between the settlers of the nascent colony of New Netherland and the native Lenape population?

Top-5 Retrieved Chunks:

Rank 1
Score    : 0.6446
Chunk ID : 5ae4c01e55429913cc2044f3_chunk_2
Sample ID: 5ae4c01e55429913cc2044f3
Title    : Kieft's War
Text     : Kieft's War, also known as the Wappinger War, was a conflict (1643–1645) between settlers of the nascent colony of New Netherland and the native Lenape population in what would later become the New York metropolitan area of the United States.  It is named for Director-General of New Netherland Wille...
--------------------------------------------------------------------------------
Rank 2
Score    : 0.5066
Chunk ID : 5a75e67d5542992db9473709_chunk_3
Sample ID: 5a75e67d5542992db9473709
Title    : Battle of Mamaroneck
Text     : The Battle of Mamaroneck was a skirmish in the New York and New Jersey campaign of the American

## Retrieval Evaluation Function

In [34]:
# ============================================================
# Cell 18 — Retrieval Evaluation Function
# ============================================================

def evaluate_retrieval(sample, retriever, top_k=5):
    """
    Evaluate retrieval recall for one HotpotQA sample.
    """

    # Retrieve documents
    results = retriever.retrieve(
        sample["question"],
        top_k=top_k
    )

    # Build gold evidence identifiers
    gold_evidence = {
        (sample["id"], title)
        for title in sample["supporting_facts"]["title"]
    }

    # Build retrieved identifiers
    retrieved_evidence = {
        (result["sample_id"], result["title"])
        for result in results
    }

    # Find retrieved gold evidence
    matched_evidence = (
        gold_evidence.intersection(retrieved_evidence)
    )

    # Calculate recall
    recall = (
        len(matched_evidence) / len(gold_evidence)
        if gold_evidence
        else 0.0
    )

    return {
        "sample_id": sample["id"],
        "question": sample["question"],
        "gold_evidence_count": len(gold_evidence),
        "retrieved_gold_count": len(matched_evidence),
        "recall": recall,
        "retrieval_success": len(matched_evidence) > 0,
        "results": results
    }


print("✓ Retrieval evaluation function created.")

✓ Retrieval evaluation function created.


In [35]:
# ============================================================
# Cell 19 — Evaluate Retrieval on 30 Questions
# ============================================================

retrieval_results = []

for sample in evaluation_samples:

    result = evaluate_retrieval(
        sample,
        retriever,
        top_k=DEFAULT_TOP_K
    )

    retrieval_results.append(result)


print("✓ Retrieval evaluation completed.")
print(f"Number of evaluated questions: {len(retrieval_results)}")

print("\n" + "=" * 100)

for i, result in enumerate(retrieval_results, start=1):

    print(f"\nQuestion {i}")
    print(f"Sample ID             : {result['sample_id']}")
    print(f"Question              : {result['question']}")
    print(f"Gold evidence count   : {result['gold_evidence_count']}")
    print(f"Retrieved gold count  : {result['retrieved_gold_count']}")
    print(f"Recall@{DEFAULT_TOP_K}            : {result['recall']:.2f}")
    print(f"Retrieval success     : {result['retrieval_success']}")

✓ Retrieval evaluation completed.
Number of evaluated questions: 30


Question 1
Sample ID             : 5a7a567255429941d65f25bd
Question              : What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
Gold evidence count   : 2
Retrieved gold count  : 1
Recall@5            : 0.50
Retrieval success     : True

Question 2
Sample ID             : 5abca1a55542993a06baf937
Question              : When did the park at which Tivolis Koncertsal is located open?
Gold evidence count   : 2
Retrieved gold count  : 2
Recall@5            : 1.00
Retrieval success     : True

Question 3
Sample ID             : 5a73977d554299623ed4ac08
Question              : What is the shared country of ancestry between Art Laboe and Scout Tufankjian?
Gold evidence count   : 2
Retrieved gold count  : 1
Recall@5            : 0.50
Retrieval success     : True

Question 4
Sample ID             : 5ab514c05542991779162d72
Question       

## Retrieval Summary Metrics

In [36]:
# ============================================================
# Cell 20 — Retrieval Summary Metrics
# ============================================================

recalls = [
    result["recall"]
    for result in retrieval_results
]

successful_retrievals = [
    result["retrieval_success"]
    for result in retrieval_results
]

mean_recall = sum(recalls) / len(recalls)

retrieval_success_rate = (
    sum(successful_retrievals)
    / len(successful_retrievals)
)

print("Retrieval Evaluation Summary")
print("=" * 60)

print(f"Number of questions : {len(retrieval_results)}")
print(f"Top-k               : {DEFAULT_TOP_K}")

print(f"\nMean Recall@{DEFAULT_TOP_K}       : {mean_recall:.4f}")
print(f"Retrieval Success Rate : {retrieval_success_rate:.4f}")

print("\nRecall distribution:")

for recall_value in sorted(set(recalls), reverse=True):
    count = recalls.count(recall_value)

    print(
        f"  Recall@{DEFAULT_TOP_K} = "
        f"{recall_value:.2f} : {count} questions"
    )

Retrieval Evaluation Summary
Number of questions : 30
Top-k               : 5

Mean Recall@5       : 0.6500
Retrieval Success Rate : 0.9000

Recall distribution:
  Recall@5 = 1.00 : 12 questions
  Recall@5 = 0.50 : 15 questions
  Recall@5 = 0.00 : 3 questions
